In [3]:
import pandas as pd
import glob

files = glob.glob("sap-o2c-data/outbound_delivery_headers/*.jsonl")

df_delivery = pd.concat(
    [pd.read_json(f, lines=True) for f in files],
    ignore_index=True
)

print(df_delivery.head())

  actualGoodsMovementDate                   actualGoodsMovementTime  \
0                    None  {'hours': 0, 'minutes': 0, 'seconds': 0}   
1                    None  {'hours': 0, 'minutes': 0, 'seconds': 0}   
2                    None  {'hours': 0, 'minutes': 0, 'seconds': 0}   
3                    None  {'hours': 0, 'minutes': 0, 'seconds': 0}   
4                    None  {'hours': 0, 'minutes': 0, 'seconds': 0}   

               creationDate                                creationTime  \
0  2025-03-31T00:00:00.000Z  {'hours': 6, 'minutes': 49, 'seconds': 13}   
1  2025-03-31T00:00:00.000Z  {'hours': 6, 'minutes': 49, 'seconds': 16}   
2  2025-03-31T00:00:00.000Z  {'hours': 11, 'minutes': 19, 'seconds': 3}   
3  2025-04-02T00:00:00.000Z   {'hours': 5, 'minutes': 1, 'seconds': 59}   
4  2025-04-02T00:00:00.000Z   {'hours': 5, 'minutes': 1, 'seconds': 59}   

  deliveryBlockReason  deliveryDocument hdrGeneralIncompletionStatus  \
0                              80737721           

In [4]:
df_delivery = df_delivery.rename(columns={
    "deliveryDocument": "delivery_id",
    "creationDate": "delivery_date",
    "shippingPoint": "shipping_point"
})

In [5]:
df_delivery = df_delivery[[
    "delivery_id",
    "delivery_date",
    "shipping_point"
]]

In [6]:
import sqlite3

conn = sqlite3.connect("data.db")

df_delivery.to_sql("deliveries", conn, if_exists="replace", index=False)

print("✅ deliveries table created")

✅ deliveries table created


In [7]:
pd.read_sql("SELECT * FROM deliveries LIMIT 5", conn)

,delivery_id,delivery_date,shipping_point
0,80737721,2025-03-31T00:00:00.000Z,1920
1,80737722,2025-03-31T00:00:00.000Z,1920
2,80737723,2025-03-31T00:00:00.000Z,1920
3,80737921,2025-04-02T00:00:00.000Z,KA05
4,80737922,2025-04-02T00:00:00.000Z,MH05


In [8]:
query = """
SELECT 
    i.invoice_id,
    i.customer_id,
    ii.product_id,
    ii.delivery_id,
    d.shipping_point
FROM invoices i
JOIN invoice_items ii ON i.invoice_id = ii.invoice_id
JOIN deliveries d ON ii.delivery_id = d.delivery_id
LIMIT 5;
"""

print(pd.read_sql(query, conn))

   invoice_id  customer_id      product_id  delivery_id shipping_point
0    90504248    320000083  S8907367001003     80738072           WB05
1    90628265    320000082  S8907367013532     80754575           1301
2    90628266    320000082  S8907367002512     80754576           1301
3    90504274    320000083  S8907367003380     80738091           WB05
4    90504242    320000083  S8907367025344     80738068           WB05


In [9]:
query = """
SELECT 
    i.invoice_id,
    i.customer_id,
    ii.product_id,
    ii.delivery_id,
    d.shipping_point
FROM invoices i
JOIN invoice_items ii ON i.invoice_id = ii.invoice_id
LEFT JOIN deliveries d ON ii.delivery_id = d.delivery_id
WHERE i.invoice_id = '90504248';
"""

print(pd.read_sql(query, conn))

   invoice_id  customer_id      product_id  delivery_id shipping_point
0    90504248    320000083  S8907367001003     80738072           WB05


In [10]:
query = """
SELECT d.delivery_id
FROM deliveries d
LEFT JOIN invoice_items ii 
ON d.delivery_id = ii.delivery_id
WHERE ii.delivery_id IS NULL;
"""

print(pd.read_sql(query, conn))

   delivery_id
0     80737721
1     80737722
2     80737723
